In [1]:

# Cell 1 — rebuild the Teachable Machine model: standard MobileNetV2 + your trained layers
from pathlib import Path
import numpy as np, pandas as pd, h5py, keras
from PIL import Image, ImageOps

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_DIR = ROOT / "models" / "ok_nok"
MANIFEST  = ROOT / "data" / "ocr_frames_13_44" / "manifest.csv"
OUT       = ROOT / "data" / "ws8"
OUT.mkdir(parents=True, exist_ok=True)

# 1. read your trained layers (the "dense" weights) from the .h5 file
ws = {}
with h5py.File(MODEL_DIR / "keras_model.h5", "r") as f:
    def visit(name, obj):
        if isinstance(obj, h5py.Dataset) and name.startswith("model_weights") and "dense" in name.lower():
            ws[name] = obj[()]
    f.visititems(visit)
for k, v in ws.items():
    print(k, v.shape)

k1 = next(v for v in ws.values() if v.shape == (1280, 100))
b1 = next(v for v in ws.values() if v.shape == (100,))
k2 = next(v for v in ws.values() if v.ndim == 2 and v.shape[0] == 100)
b2 = next((v for v in ws.values() if v.ndim == 1 and v.shape[0] == k2.shape[1]),
          np.zeros(k2.shape[1]))

# 2. the standard part (same network Teachable Machine uses)
base = keras.applications.MobileNetV2(input_shape=(224, 224, 3), alpha=0.35,
                                      include_top=False, weights="imagenet")

# 3. full prediction = standard part + your trained layers
def predict(x):
    feat = base.predict(x[None], verbose=0).mean(axis=(1, 2))[0]   # 1280 features
    h = np.maximum(feat @ k1 + b1, 0)                              # your layer 1 (ReLU)
    z = h @ k2 + b2                                                # your layer 2
    e = np.exp(z - z.max())
    return e / e.sum()                                             # softmax

labels = [l.strip().split(" ", 1)[1] for l in open(MODEL_DIR / "labels.txt") if l.strip()]
print("labels:", labels)

model_weights/sequential_3/dense_Dense1/bias:0 (100,)
model_weights/sequential_3/dense_Dense1/kernel:0 (1280, 100)
model_weights/sequential_3/dense_Dense2/kernel:0 (100, 2)
2019640/2019640 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
labels: ['NOK', 'OK']


In [2]:
# Cell 2 — classify all full-bottle crops, then one decision per bottle
man = pd.read_csv(MANIFEST)

def load_img(rel_path):
    img = Image.open(ROOT / rel_path).convert("RGB")
    img = ImageOps.fit(img, (224, 224), Image.Resampling.LANCZOS)   # same as Teachable Machine
    return (np.asarray(img, dtype=np.float32) / 127.5) - 1

preds, confs = [], []
for p in man["full_crop_path"]:
    probs = predict(load_img(p))
    preds.append(labels[int(probs.argmax())].lower())
    confs.append(round(float(probs.max()), 3))
man["pred"], man["conf"] = preds, confs

# one decision per bottle: NOK if ANY camera says NOK (safe side)
cls = (man.groupby("Sequence")["pred"]
          .apply(lambda s: "nok" if (s == "nok").any() else "ok")
          .rename("AI_classification")
          .reset_index())
cls.to_csv(OUT / "classification_13_44.csv", index=False)

print("crops classified:", len(man))
print("per crop:", man["pred"].value_counts().to_dict())
print("per bottle:", cls["AI_classification"].value_counts().to_dict())
print("saved:", (OUT / "classification_13_44.csv").relative_to(ROOT))

crops classified: 394
per crop: {'ok': 339, 'nok': 55}
per bottle: {'ok': 106, 'nok': 41}
saved: data/ws8/classification_13_44.csv


In [3]:
# Cell 3 — compare a few images with the Teachable Machine website
for name in man["full_crop_path"].sample(3, random_state=1):
    print(name, dict(zip(labels, predict(load_img(name)).round(3))))

data/ocr_frames_13_44/full_13_44_541934_back_right.jpg {'NOK': np.float64(0.0), 'OK': np.float64(1.0)}
data/ocr_frames_13_44/full_13_44_541993_back_right.jpg {'NOK': np.float64(0.0), 'OK': np.float64(1.0)}
data/ocr_frames_13_44/full_13_44_542043_back_left.jpg {'NOK': np.float64(0.323), 'OK': np.float64(0.677)}
